# Greedy Knapsack Problem (Fractional)
In this lab, we explore the Fractional Knapsack Problem, a fundamental optimization problem that demonstrates the Greedy Algorithmic Paradigm.
We will implement it using greedy selection (based on value-to-weight ratio) and analyze its time and space complexity.

- Created by Dr. Ajay

# Problem Definition
Given a set of
n
n items, each with:
Value vi
Weight wi
A knapsack with maximum capacity **W**

Goal: maximize total value that can be put in the knapsack.
Constraint: You may take fractions of items.

Greedy Strategy:

*  Sort items by decreasing ratio vi/wi
*  Select items fully until knapsack is full.
*  Take fraction of the next item if capacity remains.

In [42]:
def fractional_knapsack(capacity, items):
    """
    Solves the Fractional Knapsack problem.

    Args:
        capacity (int or float): The maximum weight capacity of the knapsack.
        items (list of tuples): A list of items, where each item is a tuple
                                (value, weight).

    Returns:
        tuple: A tuple containing:
               - float: The maximum total value that can be put in the knapsack.
               - list: A list of tuples (item_index, fraction_taken) indicating
                       how much of each item was taken.
    """
    # 1. Sort items by decreasing ratio vi/wi
    # We'll store (value, weight, original_index) to keep track of items
    item_data = []
    for i, (value, weight) in enumerate(items):
        if weight > 0:
            item_data.append((value / weight, value, weight, i))
        else:
            # Handle items with zero weight (if they have positive value, they can always be taken)
            # For simplicity, if weight is 0, we can treat its ratio as infinity if value > 0
            # or 0 if value == 0. Here, we'll just skip it for ratio calculation
            # and consider it later if needed, or assume valid weights > 0.
            # For typical knapsack, weights are positive.
            pass # Or handle specific logic for zero-weight items

    item_data.sort(key=lambda x: x[0], reverse=True) # Sort by ratio (x[0])

    total_value = 0
    fractions_taken = []
    current_capacity = capacity

    for ratio, value, weight, original_index in item_data:
        if current_capacity <= 0:
            break

        if weight <= current_capacity:
            # 2. Select items fully until knapsack is full.
            total_value += value
            current_capacity -= weight
            fractions_taken.append((original_index, 1.0))
        else:
            # 3. Take fraction of the next item if capacity remains.
            fraction = current_capacity / weight
            total_value += value * fraction
            current_capacity = 0
            fractions_taken.append((original_index, fraction))

    return total_value, fractions_taken

# Example Usage:
if __name__ == '__main__':
    items = [(60, 10), (100, 20), (120, 30)]  # (value, weight)
    capacity = 50

    max_value, fractions = fractional_knapsack(capacity, items)

    print(f"Items: {items}")
    print(f"Knapsack Capacity: {capacity}")
    print(f"Maximum total value: {max_value}")
    print("Fractions taken:")
    for idx, frac in fractions:
        print(f"  Item {idx} (value={items[idx][0]}, weight={items[idx][1]}): {frac:.2f} of item")

Items: [(60, 10), (100, 20), (120, 30)]
Knapsack Capacity: 50
Maximum total value: 240.0
Fractions taken:
  Item 0 (value=60, weight=10): 1.00 of item
  Item 1 (value=100, weight=20): 1.00 of item
  Item 2 (value=120, weight=30): 0.67 of item


In this lab, we implement Huffman Coding, a greedy algorithm used for lossless data compression.
It assigns variable-length binary codes to input symbols based on their frequencies, ensuring that more frequent symbols receive shorter codes and less frequent symbols receive longer codes.

This algorithm constructs a binary tree (Huffman Tree) where each leaf node represents a character, and the path from the root to a leaf gives the character’s binary code.

Use:
Huffman Tree Construction using a Min-Priority Queue (Min-Heap).
Code generation for each character.
Encoding and decoding using the generated Huffman codes.
Complexity analysis and comparison with fixed-length encoding.

# Pseudocode:
1. Create a leaf node for each character and insert all nodes into a
2. min-heap by frequency.
   *  While the heap has more than one node:
   * Extract two nodes with the lowest frequencies.
   * Create a new internal node with frequency = sum of the two.
   *  Set the two extracted nodes as left and right children.
   *  Insert the new node back into the min-heap.
3. The remaining node is the root of the Huffman Tree.

4. Traverse the tree:
   * Assign 0 for left edge and 1 for right edge.
   * The resulting paths from root to leaves give Huffman codes.

In [43]:
import heapq
from collections import defaultdict

class HuffmanNode:
    def __init__(self, char, freq):
        self.char, self.freq, self.left, self.right = char, freq, None, None
    # Used for comparison in the min-heap
    def __lt__(self, other): return self.freq < other.freq

def build_huffman_tree(text):
    if not text: return None
    freq = defaultdict(int)
    [freq.__setitem__(c, freq[c] + 1) for c in text] # Compact frequency calculation
    pq = [HuffmanNode(char, f) for char, f in freq.items()]
    heapq.heapify(pq)

    while len(pq) > 1:
        n1, n2 = heapq.heappop(pq), heapq.heappop(pq)
        merged = HuffmanNode(None, n1.freq + n2.freq)
        merged.left, merged.right = n1, n2
        heapq.heappush(pq, merged)
    return pq[0]

def generate_huffman_codes(root):
    codes = {}
    def _traverse(node, current_code):
        if node is None: return
        if node.char is not None: codes[node.char] = current_code; return
        _traverse(node.left, current_code + '0')
        _traverse(node.right, current_code + '1')
    _traverse(root, '')
    return codes

def encode_text(text, h_codes): return ''.join(h_codes[char] for char in text)

def decode_text(encoded_text, root): # Corrected signature
    decoded_text, curr = [], root
    for bit in encoded_text:
        curr = curr.left if bit == '0' else curr.right
        if curr.char is not None:
            decoded_text.append(curr.char); curr = root
    return ''.join(decoded_text)

if __name__ == '__main__':
    sample_text = "this is an example of a huffman encoding"
    print(f"Original text: {sample_text}")

    h_root = build_huffman_tree(sample_text)
    codes = generate_huffman_codes(h_root)
    print("\nHuffman Codes:")
    for char, code in sorted(codes.items()): print(f"  '{char}': {code}")

    encoded_data = encode_text(sample_text, codes)
    print(f"\nEncoded text: {encoded_data}")
    print(f"Encoded length: {len(encoded_data)} bits")
    fixed_length_bits = len(sample_text) * 8
    print(f"Fixed-length encoding (8 bits/char): {fixed_length_bits} bits")

    decoded_data = decode_text(encoded_data, h_root)
    print(f"\nDecoded text: {decoded_data}")
    print(f"Decoded text matches original: {sample_text == decoded_data}")


Original text: this is an example of a huffman encoding

Huffman Codes:
  ' ': 110
  'a': 001
  'c': 00001
  'd': 00000
  'e': 1011
  'f': 1110
  'g': 111110
  'h': 0111
  'i': 1010
  'l': 01001
  'm': 0110
  'n': 100
  'o': 11110
  'p': 01011
  's': 0001
  't': 01010
  'u': 01000
  'x': 111111

Encoded text: 0101001111010000111010100001110001100110101111111100101100101101001101111011110111011000111001110100011101110011000110011010111000000111110000001010100111110
Encoded length: 157 bits
Fixed-length encoding (8 bits/char): 320 bits

Decoded text: this is an example of a huffman encoding
Decoded text matches original: True
